In [ ]:
import numpy as np 
import sympy as sp 
import pandas as pd
import scipy.stats as stats

import matplotlib.pyplot as plt
import seaborn as sns

Helpers

In [ ]:
# rewritten optimal version:


class RandomWalk:
    def __init__(self, p: float , domain: list[int, int]):
        self.p = p
        self.domain = domain

    # ---------
    # simulations
    # ---------

    def simulate_absorbing(self, x_0: int, traj_length: int, probe_size: int):
        # initialize on x_0
        trajectories = np.zeros((probe_size, traj_length))
        trajectories[:, 0] = x_0

        # next steps 
        for t in range(1, traj_length):
            # params 
            ## current positions 
            current_position = trajectories[:, t-1]

            # New moves 
            ## we choose at random new steps 
            moves = np.random.choice([-1, 1], size=probe_size, p=[1-self.p, self.p])
            next_moves = current_position + moves 

            # Barrier
            ## create logic mask
            is_absorbed = (current_position <= self.domain[0]) | (current_position >= self.domain[1])
            ## if all states are absorbed 
            if np.all(is_absorbed):
                trajectories[:, t:] = current_position[:, np.newaxis]
                return trajectories
            
            # update trajectories trajectories 
            trajectories[:, t] = np.where(is_absorbed, current_position, next_moves)

        return trajectories

    def simulate_elasting(self, x_0: int, traj_length: int, probe_size: int):
        # initialize on x_0
        trajectories = np.zeros((probe_size, traj_length))
        trajectories[:, 0] = x_0

        # next steps 
        for t in range(1, traj_length):
            # params 
            ## current positions 
            current_position = trajectories[:, t-1]

            # New moves 
            ## we choose at random flags (0 with 1-p, 1 with p )
            moves = np.random.choice([-1, 1], size=probe_size, p=[1 - self.p, self.p])
            ## next moves
            next_moves = current_position + moves
            ## update trajectories trajectories - we do not stop procesess we just do now allwo to exceed range
            trajectories[:, t] = np.clip(next_moves, self.domain[0], self.domain[1])
            
            # old - problem
            # trajectories[is_min: t] = np.minimum(current_position[is_min] + 1, self.domain[1])
            # trajectories[~is_min: t] = np.maximum(current_position[~is_min] - 1, self.domain[0])

        return trajectories
    
    # ---------
    # theoretical results
    # ---------

    def get_theoretical_psi_R(self, x_0_vals):
        """Calculates theoretical psi for hitting upper barrier (Right)"""
        p = self.p
        q = 1 - p
        r = q / p
        k = self.domain[1] 
        if p == 0.5:
            return x_0_vals / k
        n = x_0_vals

        numerator = (r**x_0_vals) - 1
        denominator = (r**k) - 1
        
        return numerator / denominator
    
    def get_theoretical_stationary_dist(self, x_vals):
        """Calculates theoretical stationary distribution for elastic barriers"""
        p = self.p
        k = self.domain[1]
        
        if p == 0.5:
            # Uniform distribution: 1 / (number of states)
            return np.full_like(x_vals, 1.0 / (k + 1), dtype=float)
        
        r = p / (1 - p)
        # Formula: pi_i = (r**i * (1 - r)) / (1 - r**(k + 1))
        numerator = (r**x_vals) * (1 - r)
        denominator = 1 - r**(k + 1)
        
        return numerator / denominator


### Random walks with absorbing barriers


#### Theoretical solution 

##### First task (probability of hitting barrier)

We assume that available states are $n \in \{0 , \ldots, k\}$. We take $0<n<k$ and set $R_n$ as probability of hitting $k$ when starting at $n$, from that we obtain:
$$R_{n} = pR_{n+1} + (1-p)R_{n-1}$$
We create recurrence relation:
$$pR_{n+1} - R_{n}+(1-p)R_{n-1}=0$$
We put our final solution to be $R_{n}=\lambda^{n}$:
$$p\lambda^{2}-\lambda+(1-p)=0$$
We solve for $\lambda$
$$\Delta=1 - 4p(1-p) \implies \lambda_{1,2}= \frac{{1\pm \sqrt{ 1-4p(1-p) }}}{2p} = \frac{{1\pm 2(p-1/2)}}{2p}$$
We obtain solution of form:
$$\begin{align}
\lambda_{1} & =1  & \lambda_{2} & = \frac{1-p}{p}  & R_{n}=a\cdot\left( \frac{1-p}{p} \right)^{n} + b
\end{align}$$
We obtain $a$ and $b$ by solving it for initial conditions
$$\begin{cases}
0  & = a\cdot\left[ 1=\left( \frac{1-p}{p} \right)^{n=0} \right] + b \\
1 & =a\cdot\left( \frac{1-p}{p} \right)^{k} + b
\end{cases}
\Leftrightarrow \begin{cases}
a  & = \frac{1}{\left( \frac{1-p}{p} \right)^{k} -1} \\
b  & = -\frac{1}{\left( \frac{1-p}{p} \right)^{k} -1}
\end{cases}$$
Finally we obtain:
$$R_{n} = \frac{1}{\left( \frac{1-p}{p} \right)^{k} -1} \cdot\left( \left( \frac{1-p}{p} \right)^{n}- 1 \right)$$
To obtain $L_n$ we just need to swap $p$ and $1-p$ ro calculate 1 - result (because with probability 1 we hit wall)

##### Stationary distribution 

To obtain stationary distribution $\pi$ we need to find $pi_i$ such that:
$$\pi_i \cdot P_{i, i+1}=\pi_{i+1} \cdot P_{i+1, i}$$
Which is equivalent to:
$$\pi_i \cdot p = \pi_{i+1} \cdot(1-p) \implies \pi_{i+1} = \pi_i \cdot \frac{p}{1-p} = \pi_0 \cdot r^i$$
Where $r= \frac{p}{1-p}$, if we normalize everything we finally obtain:
$$\pi_i = r^i\frac{(1-r)}{1-r^{k+1}}$$



In [ ]:
# params
## experiment settings
p = 0.4
barrier = [0, 20]
traj_length = 300 # calculate expected value of hittingh 
probe_size = 10_00 # 
## helper variable (domain)
x0_range = range(barrier[0], barrier[1]+1)

## define random walk
random_walk = RandomWalk(p=p, domain=barrier)

emp_psi_0 = []
emp_psi_k = []

for x_0 in x0_range:
    probe = random_walk.simulate_absorbing(x_0, traj_length=traj_length, probe_size=probe_size)
    final_states = probe[:,-1]

    emp_psi_0.append(np.mean(final_states <= 0)) 
    emp_psi_k.append(np.mean(final_states >= barrier[1]))


In [ ]:
def plot_trajectories_barriers(trajectories: np.array, barrier: list[int, int], plt_title: str):
    # --- Parameters ---
    probe_size = trajectories.shape[0]

    # --- Plotting ---
    plt.figure(figsize=(12, 6))

    # Plot each individual trajectory
    for i in range(probe_size):
        plt.plot(trajectories[i, :], alpha=0.7, linewidth=1.5, label=f'Sample {i+1}' if probe_size <= 5 else None)

    # Add horizontal lines for the boundaries
    plt.axhline(y=barrier[0], color='red', linestyle='--', label='Lower Barrier')
    plt.axhline(y=barrier[1], color='red', linestyle='--', label='Upper Barrier')

    # Plot formatting
    plt.title(plt_title, fontsize=14)
    plt.xlabel('Time (Step)', fontsize=12)
    plt.ylabel('Position (State)', fontsize=12)
    plt.grid(True, linestyle=':', alpha=0.6)

    # Set Y-axis limits slightly wider than the domain for better visibility
    plt.ylim(barrier[0] - 1, barrier[1] + 1) 

    # Add legend if the number of samples is small
    if probe_size <= 10:
        plt.legend(loc='upper left', bbox_to_anchor=(1, 1))

    plt.tight_layout()
    plt.show()

In [ ]:
# --- Initialization of parameters ---
p = 0.4          # probability of stepping right
barrier = [0, 20]   # boundaries (min, max)
x_0 = 10           # starting position
traj_length = 100  # number of time steps
probe_size = 10    # number of simulated trajectories (samples)

# --- Object creation and simulation ---
rw = RandomWalk(p=p, domain=barrier)
# Choose between simulate_absorbing or simulate_elasting
trajectories = rw.simulate_absorbing(x_0, traj_length, probe_size) 

# --- Plotting ---
plt.figure(figsize=(12, 6))

# Plot each individual trajectory
for i in range(probe_size):
    plt.plot(trajectories[i, :], alpha=0.7, linewidth=1.5, label=f'Sample {i+1}' if probe_size <= 5 else None)

# Add horizontal lines for the boundaries
plt.axhline(y=barrier[0], color='red', linestyle='--', label='Lower Barrier')
plt.axhline(y=barrier[1], color='red', linestyle='--', label='Upper Barrier')

# Plot formatting
plt.title(f'Random Walk Trajectories ($p={p}$)', fontsize=14)
plt.xlabel('Time (Step)', fontsize=12)
plt.ylabel('Position (State)', fontsize=12)
plt.grid(True, linestyle=':', alpha=0.6)

# Set Y-axis limits slightly wider than the domain for better visibility
plt.ylim(barrier[0] - 1, barrier[1] + 1) 

# Add legend if the number of samples is small
if probe_size <= 10:
    plt.legend(loc='upper left', bbox_to_anchor=(1, 1))

plt.tight_layout()
plt.show()

In [ ]:
# plot - probabilities 
x0_range = np.arange(barrier[0], barrier[1]+1)

theoretical_k = random_walk.get_theoretical_psi_R(x0_range)

# plot
plt.figure(figsize=(10, 6))
plt.plot(x0_range, emp_psi_k, 'ro', label='Empirical $\psi_k$ (MC)', alpha=0.6)
plt.plot(x0_range, theoretical_k, 'k-', label='Theoretical $\psi_k$')
plt.plot(x0_range, emp_psi_0, 'bs', label='Empirical $\psi_0$ (MC)', alpha=0.6)
plt.plot(x0_range, 1 - theoretical_k, 'k--', label='Theoretical $\psi_0$')
# plot settings
plt.title(f'Probability of absorptions ($p={p}, k={random_walk.domain[1]}$)')
plt.xlabel('Inital point $x_0$')
plt.ylabel('Probability')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

Random walks with elastic barrier

In [ ]:
# params
## experiment settings
p = 0.4
barrier = [0, 20]
traj_length = 10_00
probe_size = 10_00

## define random walk
random_walk = RandomWalk(p=p, domain=barrier)
result = random_walk.simulate_elasting(3, traj_length=traj_length, probe_size=probe_size)

In [ ]:

# Theoretical calculations
x_range = np.arange(barrier[0], barrier[1] + 1)
theo_stationary = random_walk.get_theoretical_stationary_dist(x_range)

# Plotting
plt.figure(figsize=(12, 6))

# Plot empirical distribution from simulation
# Using bins=np.arange(k+2)-0.5 to center bars over integer ticks
sns.histplot(result[:, -1], stat='probability', bins=np.arange(k+2)-0.5, 
             label='Empirical (MC)', alpha=0.6, color='skyblue')

# Overlay theoretical stationary distribution
plt.step(x_range, theo_stationary, where='mid', color='red', 
         label='Theoretical Stationary Dist.', linewidth=2)
plt.plot(x_range, theo_stationary, 'ro', markersize=4)

# Formatting
plt.xticks(x_range)
plt.title(f'Stationary Distribution for Elastic Barriers ($p={p}, k={k}$)')
plt.xlabel('State $x$')
plt.ylabel('Probability $\pi(x)$')
plt.legend()
plt.grid(True, alpha=0.2)
plt.show()

Muszę zrobić tak (to wykładu)
- metryki oraz ich interpertacje w przypadku danych kliniczych (wogóle więcej się nad nimi pocylić) 
- PRS skąd jest populacja brana
-  